# Fig. 4 - Combined R_x(beta) + R_x(delta) intervention across the four game regimes.

Row A: T = 15 days, row B: T = 30 days. Columns: Snowdrift, Harmony, Coordination,
Prisoner's Dilemma (alpha = 0.4, delta = 0.6). In each regime the upper map is tumor
reduction R_N(T) and the lower map leader suppression S_L(T) = 1 - x(T), both in %,
over the dose grid (d_beta on x, d_delta on y). The red square marks the dose pair
that maximizes J = (R_N/100 + S_L)/2.
Output: ../figures/Fig4_combination_Rbeta_Rdelta_landscapes.png / .pdf

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from lf_model import REGIMES, REGIME_ORDER, treatment_outcome, set_style, save

set_style()
doses = np.linspace(0, 1, 21)                    # dose step 0.05
DB, DD = np.meshgrid(doses, doses)               # arrays indexed [d_delta, d_beta]
maps = {}
for T in (15.0, 30.0):
    for name, (b, g) in REGIMES.items():
        r_n, s_l, J, _ = treatment_outcome(b, g, {"beta": DB, "delta": DD}, T=T)
        iy, ix = np.unravel_index(np.argmax(J), J.shape)
        maps[T, name] = (np.clip(r_n, 0, 100), 100 * s_l, (iy, ix))
        print(f"T={T:.0f} {name:20s} best (d_beta, d_delta) = ({doses[ix]:.2f}, {doses[iy]:.2f}): "
              f"R_N = {r_n[iy, ix]:.1f}%, S_L = {s_l[iy, ix]:.2f}, J = {J[iy, ix]:.3f}")

T=15 Snowdrift            best (d_beta, d_delta) = (1.00, 0.50): R_N = 99.3%, S_L = 0.99, J = 0.992
T=15 Harmony              best (d_beta, d_delta) = (1.00, 0.40): R_N = 92.0%, S_L = 0.96, J = 0.940


T=15 Coordination         best (d_beta, d_delta) = (1.00, 0.40): R_N = 94.1%, S_L = 0.96, J = 0.951
T=15 Prisoner's Dilemma   best (d_beta, d_delta) = (1.00, 0.55): R_N = 98.3%, S_L = 0.99, J = 0.985


T=30 Snowdrift            best (d_beta, d_delta) = (1.00, 0.75): R_N = 99.6%, S_L = 1.00, J = 0.996


T=30 Harmony              best (d_beta, d_delta) = (1.00, 0.55): R_N = 98.7%, S_L = 0.99, J = 0.990


T=30 Coordination         best (d_beta, d_delta) = (1.00, 0.55): R_N = 99.0%, S_L = 0.99, J = 0.991


T=30 Prisoner's Dilemma   best (d_beta, d_delta) = (1.00, 0.75): R_N = 99.6%, S_L = 1.00, J = 0.996


In [2]:
step = doses[1] - doses[0]
ext = [-step / 2, 1 + step / 2, -step / 2, 1 + step / 2]
CMAP = "magma"
fig = plt.figure(figsize=(14.5, 12.2))
outer = GridSpec(2, 1, figure=fig, left=0.08, right=0.82, top=0.93, bottom=0.075, hspace=0.30)
im = None
for row, T in enumerate((15.0, 30.0)):
    cols = outer[row].subgridspec(1, 4, wspace=0.12)
    first_ax = None
    for j, name in enumerate(REGIME_ORDER):
        r_n, s_l, (iy, ix) = maps[T, name]
        pair = cols[j].subgridspec(2, 1, hspace=0.04)
        for k, arr in enumerate((r_n, s_l)):
            ax = fig.add_subplot(pair[k])
            im = ax.imshow(arr, origin="lower", extent=ext, cmap=CMAP, vmin=0, vmax=100,
                           interpolation="nearest", aspect="auto")
            ax.plot(doses[ix], doses[iy], "s", ms=11, mfc="none", mec="#00e5ff", mew=2.4, clip_on=False, zorder=5)
            ax.set_xticks([0, 0.5, 1]); ax.set_yticks([0, 0.5, 1])
            ax.tick_params(labelsize=10, length=3)
            if k == 0:
                ax.tick_params(labelbottom=False)
            if j > 0:
                ax.tick_params(labelleft=False)
            for sp in ax.spines.values():
                sp.set_visible(True); sp.set_linewidth(0.8)
            if row == 0 and k == 0:
                ax.set_title(name, fontsize=14, fontweight="bold", pad=8)
            if j == 3:          # name the two maps once, on the right of the row
                ax.text(1.07, 0.5, ["Tumor\nreduction $R_N$", "Leader\nsuppression $S_L$"][k],
                        transform=ax.transAxes, rotation=0, ha="left", va="center", fontsize=11.5)
            if j == 0 and k == 0:
                first_ax = ax
    # one y-label and one x-label per row
    bb = outer[row].get_position(fig)
    fig.text(bb.x0 - 0.05, (bb.y0 + bb.y1) / 2, r"$R_x\delta$ dose, $d_\delta$",
             rotation=90, ha="center", va="center", fontsize=15)
    fig.text((bb.x0 + bb.x1) / 2, bb.y0 - 0.045, r"$R_x\beta$ dose, $d_\beta$",
             ha="center", va="center", fontsize=15)
    fig.text(bb.x0 - 0.065, bb.y1 + (0.035 if row == 0 else 0.012), "AB"[row],
             fontsize=22, fontweight="bold", ha="left", va="bottom")
    fig.text(bb.x0 - 0.035, bb.y1 + (0.035 if row == 0 else 0.012), f"$T = {T:.0f}$ days",
             fontsize=14, ha="left", va="bottom")
cax = fig.add_axes([0.925, 0.30, 0.013, 0.40])
cb = fig.colorbar(im, cax=cax)
cb.set_label("Tumor reduction or leader suppression (%)", fontsize=12)
cb.ax.tick_params(labelsize=10)
save(fig, "Fig4_combination_Rbeta_Rdelta_landscapes", outdir="../figures")
plt.close(fig)

Saved ../figures/Fig4_combination_Rbeta_Rdelta_landscapes.png/.pdf
